In [4]:
# 📓 Feature Engineering Notebook 
import pandas as pd
import numpy as np
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder

# ---
# Load Cleaned Data
df = pd.read_csv("../data/processed/twitter_messages_processed_v1.csv")
df.head()


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,sentiment
0,700,115854,True,2017-10-31 22:16:56+00:00,@applesupport why are my i️’s changing not sho...,698,NaN,positive
1,714,115856,True,2017-10-31 22:19:32+00:00,hey @applesupport and anyone else who upgraded...,"712,715",NaN,neutral
2,723,115859,True,2017-10-31 22:11:16+00:00,@115858 @applesupport hello are all the lines ...,722,NaN,negative
3,730,115861,True,2017-10-31 20:46:35+00:00,"hello, internet. can someone explain why this ...","729,731",NaN,neutral
4,733,115863,True,2017-10-31 22:16:40+00:00,@applesupport i’ve got a screenshot saying my ...,732,NaN,neutral


In [5]:
df['created_at'] = pd.to_datetime(df['created_at'], utc=True)

# Extract date-based features
df['tweet_date'] = df['created_at'].dt.date
df['tweet_year'] = df['created_at'].dt.year
df['tweet_month'] = df['created_at'].dt.month
df['tweet_day'] = df['created_at'].dt.day
df['tweet_weekday'] = df['created_at'].dt.weekday  # Monday=0, Sunday=6
df['tweet_hour'] = df['created_at'].dt.hour
df['tweet_quarter'] = df['created_at'].dt.quarter

print(df[['created_at', 'tweet_year', 'tweet_month', 'tweet_day', 'tweet_weekday', 'tweet_hour', 'tweet_quarter']].head())


                 created_at  tweet_year  tweet_month  tweet_day  \
0 2017-10-31 22:16:56+00:00        2017           10         31   
1 2017-10-31 22:19:32+00:00        2017           10         31   
2 2017-10-31 22:11:16+00:00        2017           10         31   
3 2017-10-31 20:46:35+00:00        2017           10         31   
4 2017-10-31 22:16:40+00:00        2017           10         31   

   tweet_weekday  tweet_hour  tweet_quarter  
0              1          22              4  
1              1          22              4  
2              1          22              4  
3              1          20              4  
4              1          22              4  


In [6]:
df

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,sentiment,tweet_date,tweet_year,tweet_month,tweet_day,tweet_weekday,tweet_hour,tweet_quarter
0,700,115854,True,2017-10-31 22:16:56+00:00,@applesupport why are my i️’s changing not sho...,698,NaN,positive,2017-10-31,2017,10,31,1,22,4
1,714,115856,True,2017-10-31 22:19:32+00:00,hey @applesupport and anyone else who upgraded...,"712,715",NaN,neutral,2017-10-31,2017,10,31,1,22,4
2,723,115859,True,2017-10-31 22:11:16+00:00,@115858 @applesupport hello are all the lines ...,722,NaN,negative,2017-10-31,2017,10,31,1,22,4
3,730,115861,True,2017-10-31 20:46:35+00:00,"hello, internet. can someone explain why this ...","729,731",NaN,neutral,2017-10-31,2017,10,31,1,20,4
4,733,115863,True,2017-10-31 22:16:40+00:00,@applesupport i’ve got a screenshot saying my ...,732,NaN,neutral,2017-10-31,2017,10,31,1,22,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50833,2987482,823733,True,2017-11-22 00:40:57+00:00,@applesupport why is my iphone 7 constantly se...,2987481,NaN,neutral,2017-11-22,2017,11,22,2,0,4
50834,2987605,689907,True,2017-11-22 02:11:43+00:00,hey @applesupport - not being able to duplicat...,2987604,NaN,negative,2017-11-22,2017,11,22,2,2,4
50835,2987607,823765,True,2017-11-22 02:17:14+00:00,yo @applesupport is that weird glitch w/ the c...,2987606,NaN,negative,2017-11-22,2017,11,22,2,2,4
50836,2987663,823779,True,2017-11-22 03:24:02+00:00,what the fuck @applesupport my phone keeps ha...,2987662,NaN,negative,2017-11-22,2017,11,22,2,3,4


In [7]:
# Length of tweet text in characters
df['text_length'] = df['text'].str.len()

# Number of words in tweet text
df['word_count'] = df['text'].str.split().apply(len)

print(df[['text_length', 'word_count']].describe())


        text_length    word_count
count  50838.000000  50838.000000
mean     120.249498     20.281325
std       47.999777      9.068499
min       13.000000      1.000000
25%       87.000000     14.000000
50%      119.000000     20.000000
75%      139.000000     24.000000
max      328.000000     68.000000


In [8]:
df

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,sentiment,tweet_date,tweet_year,tweet_month,tweet_day,tweet_weekday,tweet_hour,tweet_quarter,text_length,word_count
0,700,115854,True,2017-10-31 22:16:56+00:00,@applesupport why are my i️’s changing not sho...,698,NaN,positive,2017-10-31,2017,10,31,1,22,4,124,18
1,714,115856,True,2017-10-31 22:19:32+00:00,hey @applesupport and anyone else who upgraded...,"712,715",NaN,neutral,2017-10-31,2017,10,31,1,22,4,136,25
2,723,115859,True,2017-10-31 22:11:16+00:00,@115858 @applesupport hello are all the lines ...,722,NaN,negative,2017-10-31,2017,10,31,1,22,4,70,11
3,730,115861,True,2017-10-31 20:46:35+00:00,"hello, internet. can someone explain why this ...","729,731",NaN,neutral,2017-10-31,2017,10,31,1,20,4,162,25
4,733,115863,True,2017-10-31 22:16:40+00:00,@applesupport i’ve got a screenshot saying my ...,732,NaN,neutral,2017-10-31,2017,10,31,1,22,4,131,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50833,2987482,823733,True,2017-11-22 00:40:57+00:00,@applesupport why is my iphone 7 constantly se...,2987481,NaN,neutral,2017-11-22,2017,11,22,2,0,4,98,18
50834,2987605,689907,True,2017-11-22 02:11:43+00:00,hey @applesupport - not being able to duplicat...,2987604,NaN,negative,2017-11-22,2017,11,22,2,2,4,150,24
50835,2987607,823765,True,2017-11-22 02:17:14+00:00,yo @applesupport is that weird glitch w/ the c...,2987606,NaN,negative,2017-11-22,2017,11,22,2,2,4,107,20
50836,2987663,823779,True,2017-11-22 03:24:02+00:00,what the fuck @applesupport my phone keeps ha...,2987662,NaN,negative,2017-11-22,2017,11,22,2,3,4,90,15


In [9]:
# Drop columns that are IDs or text columns (if not used in ML directly)
cols_to_drop = ['tweet_id', 'author_id', 'created_at', 
                'text', 'response_tweet_id', 'in_response_to_tweet_id', 'tweet_date',
               'inbound']

df= df.drop(columns=cols_to_drop)

print("Final dataset for ML shape:", df.shape)
print("Columns:", df.columns.tolist())


Final dataset for ML shape: (50838, 9)
Columns: ['sentiment', 'tweet_year', 'tweet_month', 'tweet_day', 'tweet_weekday', 'tweet_hour', 'tweet_quarter', 'text_length', 'word_count']


In [10]:
# -------------------------------------------------------------
print("\n💾 Saving cleaned version to interim file (optional step)...")

df.to_csv("../data/processed/twitter_messages_processed_v1.csv", index=False)
print("feature engineering preprocessing complete!")


💾 Saving cleaned version to interim file (optional step)...
feature engineering preprocessing complete!


In [11]:
label_encoder = LabelEncoder()
df['sentiment'] = label_encoder.fit_transform(df['sentiment'])

In [12]:
print(label_encoder.classes_)

['negative' 'neutral' 'positive']
